# 통신 데이터 통합 및 전처리

## 1. 파일 로드 및 환경 설정

In [1]:
import os
print(os.getcwd())
print(os.listdir())

c:\Users\wlfbs\final_project\Jiryun
['test1321.ipynb']


In [ ]:
import os
import glob
import duckdb
import pandas as pd

DATA_DIR = "../data"
TEMP_DIR = "../duck_temp"

con = duckdb.connect()
con.execute("SET threads=2;")
con.execute(f"SET temp_directory='{TEMP_DIR}';")

## 2. parquet 통합 및 저장

In [ ]:
# t25: CSV → Parquet 변환 (최종 분석용 아님, 저장 효율 개선 목적)

input_file = r"../data/t25_2023_2025_all.csv"
output_file = r"../data/t25_2023_2025_all.parquet"

con.execute(f"""
COPY (
    SELECT *
    FROM read_csv_auto('{input_file}', header=True)
)
TO '{output_file}'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ t25 parquet 변환 완료 (CSV → parquet)")

✅ parquet 변환 완료


이코드를 이름만바꿔 변환 완료(최종 파일은 parquet) 이미 만든 원본 데이터를 합쳐서 만든 csv t25 all은 parquet로 변환

In [ ]:
# 잘 변환됬는지 확인 코드
import pyarrow.parquet as pq

file_path = "../data/t25_2023_2025_all.parquet"
parquet_file = pq.ParquetFile(file_path)

print(parquet_file.schema.names)

['ETL_YMD', 'DOW', 'O_TIME_CD', 'O_CTY_CD', 'O_MEGA_NM', 'O_CTY_NM', 'O_CENTER_X', 'O_CENTER_Y', 'D_TIME_CD', 'D_CTY_CD', 'D_MEGA_NM', 'D_CTY_NM', 'D_CENTER_X', 'D_CENTER_Y', 'PURPOSE', 'TRANS_GB', 'SEX_CD', 'AGE_GRP', 'CNT']


# 3. 통신 원본 데이터 통합 및 저장
- t4 ~ t27 (t25 제외) 데이터를 월별·연도별 원본 파일에서 통합하여 parquet로 저장
- 본 섹션에서는 파일 생성(통합 및 저장)만 수행하며, 검증은 이후 섹션에서 수행.

In [ ]:
# 1. 파일 찾기
files = sorted(glob.glob("../data/T26_2025*.csv"))
print("찾은 파일 수:", len(files))
print(files[:3])  # 일부만 확인

# 파일 없으면 바로 중단
if len(files) == 0:
    raise ValueError("파일을 못 찾았습니다. 경로 또는 파일명 확인하세요.")

# 2. 하나씩 읽어서 리스트에 저장
df_list = []

for f in files:
    print("읽는 중:", f)
    
    try:
        temp = pd.read_csv(f, encoding="cp949")
    except UnicodeDecodeError:
        temp = pd.read_csv(f, encoding="utf-8-sig")
    
    df_list.append(temp)

# 3. 합치기
df_2025 = pd.concat(df_list, ignore_index=True)

print("합친 후 shape:", df_2025.shape)

# 4. parquet로 저장
df_2025.to_parquet("../data/t26_2025_all.parquet", index=False)

print("저장 완료: t26_2025_all.parquet")

찾은 파일 수: 12
['../data\\T26_202501_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv', '../data\\T26_202502_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv', '../data\\T26_202503_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv']
읽는 중: ../data\T26_202501_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
읽는 중: ../data\T26_202502_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
읽는 중: ../data\T26_202503_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
읽는 중: ../data\T26_202504_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
읽는 중: ../data\T26_202505_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
읽는 중: ../data\T26_202506_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
읽는 중: ../data\T26_202507_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
읽는 중: ../data\T26_202508_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
읽는 중: ../data\T26_202509_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
읽는 중: ../data\T26_202510_GG_PURPOSE_TRANS_SEXAGE_DURATION

In [ ]:
con.execute("""
SELECT ETL_YMD, COUNT(*) AS cnt
FROM read_parquet('../data/t26_2023_all.parquet')
GROUP BY ETL_YMD
ORDER BY ETL_YMD
""").fetchdf()

,ETL_YMD,cnt
0,20230101,162212
1,20230102,203080
2,20230103,208617
3,20230104,213172
4,20230105,212303
...,...,...
360,20231227,228803
361,20231228,227208
362,20231229,234839
363,20231230,177103


In [ ]:
print(con.execute("""
SELECT COUNT(*) 
FROM read_parquet('../data/t27_2025_all.parquet')
""").fetchall())

[(112325802,)]


In [9]:
print(con.execute("""
SELECT SUBSTR(CAST(ETL_YMD AS VARCHAR), 1, 6) AS ym, COUNT(*) AS cnt
FROM read_parquet('../data/t27_2025_all.parquet')
GROUP BY ym
ORDER BY ym
""").fetchdf())

        ym      cnt
0   202501  8603725
1   202502  8291492
2   202503  9372605
3   202504  9342002
4   202505  9599779
5   202506  9384291
6   202507  9838023
7   202508  9620989
8   202509  9596076
9   202510  9444539
10  202511  9495772
11  202512  9736509


### 3-1) 월별로 합친 데이터 통합을 년도로 묶어서 통합

In [ ]:
# 데이터 이미 합쳤는데 돌려서 생긴 오류
con.execute("""
COPY (
    SELECT * FROM read_parquet('../data/t26_2023_all.parquet')
    UNION ALL
    SELECT * FROM read_parquet('../data/t26_2024_all.parquet')
    UNION ALL
    SELECT * FROM read_parquet('../data/t26_2025_all_clean.parquet')
)
TO '../data/t26_2023_2025_all.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ 최종 통합 완료")

OutOfMemoryException: Out of Memory Error: Allocation failure

### 3-2) 원본 파일중 공백표시된 데이터로 인해 공백제거 후 통합

In [ ]:
files = sorted(glob.glob(r"../data/T26_2025*.csv"))
os.makedirs("../data/t26_2025_clean", exist_ok=True)

for f in files:
    print("처리 중:", f)
    
    out_path = os.path.join("../data/t26_2025_clean", os.path.basename(f))
    
    with open(f, "r", encoding="utf-8") as fr, open(out_path, "w", encoding="utf-8-sig") as fw:
        # 첫 줄(헤더)만 읽어서 공백 제거
        header = fr.readline().strip("\n")
        clean_header = ",".join([col.strip() for col in header.split(",")])
        fw.write(clean_header + "\n")
        
        # 나머지 데이터는 그대로 복사
        for line in fr:
            fw.write(line)
    
    print("저장 완료:", out_path)

print("✅ 헤더 공백 제거 파일 생성 완료")

처리 중: ../data\T26_202501_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
저장 완료: ../data/t26_2025_clean\T26_202501_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
처리 중: ../data\T26_202502_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
저장 완료: ../data/t26_2025_clean\T26_202502_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
처리 중: ../data\T26_202503_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
저장 완료: ../data/t26_2025_clean\T26_202503_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
처리 중: ../data\T26_202504_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
저장 완료: ../data/t26_2025_clean\T26_202504_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
처리 중: ../data\T26_202505_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
저장 완료: ../data/t26_2025_clean\T26_202505_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
처리 중: ../data\T26_202506_GG_PURPOSE_TRANS_SEXAGE_DURATION_ADMI_INFLOW_성남시.csv
저장 완료: ../data/t26_2025_clean\T26_202506_GG_PURPOSE_TRANS_SEXA

In [ ]:
# t26의 2025파일 중 컬럼 불일치가 있어 코드 재확인
files = sorted(glob.glob(r"../data/t26_2025_clean/T26_2025*.csv"))
file_list_sql = ", ".join([f"'{f}'" for f in files])

con.execute(f"""
COPY (
    SELECT *
    FROM read_csv_auto(
        [{file_list_sql}],
        header=true,
        union_by_name=true,
        ignore_errors=true
    )
)
TO '../data/t26_2025_all_clean.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ 2025 clean parquet 저장 완료")

✅ 2025 clean parquet 저장 완료


In [ ]:
# 컬럼 확인
df_cols = con.execute("""
DESCRIBE SELECT * FROM read_parquet('../data/t26_2025_all_clean.parquet')
""").df()

print(df_cols['column_name'].tolist())
print("컬럼 수:", len(df_cols))

['ETL_YMD', 'DOW', 'D_TIME_CD', 'D_ADMI_CD', 'D_MEGA_NM', 'D_CTY_NM', 'D_ADMI_NM', 'D_CENTER_X', 'D_CENTER_Y', 'PURPOSE', 'TRANS_GB', 'DURATION', 'SEX_CD', 'AGE_GRP', 'CNT']
컬럼 수: 15


In [ ]:
# 최종 코드 컬럼 불일치로 인한 2023~2025 컬럼 통일 후 union all
con.execute("""
COPY (
    SELECT 
        ETL_YMD, DOW, D_TIME_CD, D_ADMI_CD, D_MEGA_NM, D_CTY_NM, D_ADMI_NM,
        D_CENTER_X, D_CENTER_Y, PURPOSE, TRANS_GB, DURATION, SEX_CD, AGE_GRP, CNT
    FROM read_parquet('../data/t26_2023_all.parquet')

    UNION ALL

    SELECT 
        ETL_YMD, DOW, D_TIME_CD, D_ADMI_CD, D_MEGA_NM, D_CTY_NM, D_ADMI_NM,
        D_CENTER_X, D_CENTER_Y, PURPOSE, TRANS_GB, DURATION, SEX_CD, AGE_GRP, CNT
    FROM read_parquet('../data/t26_2024_all.parquet')

    UNION ALL

    SELECT 
        ETL_YMD, DOW, D_TIME_CD, D_ADMI_CD, D_MEGA_NM, D_CTY_NM, D_ADMI_NM,
        D_CENTER_X, D_CENTER_Y, PURPOSE, TRANS_GB, DURATION, SEX_CD, AGE_GRP, CNT
    FROM read_parquet('../data/t26_2025_all_clean.parquet')
)
TO '../data/t26_2023_2025_all_fixed.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ 최종 통합 완료")

✅ 최종 통합 완료


In [ ]:
# 나머지 파일들 이코드로 파일명만 바꿔 년도 통합
con.execute("SET threads=2;")
con.execute("SET temp_directory='../duck_temp';")

con.execute("""
COPY (
    SELECT *
    FROM read_csv_auto(
        '../data/T24_GG_PURPOSE_ADMI_POP_*.csv',
        header=true,
        union_by_name=true,
        ignore_errors=true
    )
)
TO '../data/t24_2023_2025_all.parquet'
(FORMAT PARQUET, COMPRESSION ZSTD);
""")

print("✅ T24 2023~2025 통합 완료")

✅ T24 2023~2025 통합 완료


### 같은 데이터가 있어 합칠려는 중 날짜의 타입이 다르다고 오류가 떠 코드 지운 후 날짜 컬럼 데이터 타입 확인

In [ ]:
# 날짜 컬럼 데이터 타입 확인
print(con.execute("""
DESCRIBE SELECT * 
FROM read_parquet('../data/t24_2023_2025_all.parquet')
""").df())

  column_name column_type null   key default extra
0     ETL_YMD      BIGINT  YES  None    None  None
1      CTY_CD      BIGINT  YES  None    None  None
2     MEGA_NM     VARCHAR  YES  None    None  None
3      CTY_NM     VARCHAR  YES  None    None  None
4      SEX_CD     VARCHAR  YES  None    None  None
5     AGE_GRP     VARCHAR  YES  None    None  None
6     TIME_CD     VARCHAR  YES  None    None  None
7     PURPOSE      BIGINT  YES  None    None  None
8         CNT      DOUBLE  YES  None    None  None


In [ ]:
# 컬럼 수 확인
print(con.execute("""
SELECT COUNT(*) AS cnt
FROM read_parquet('../data/t22_2023_2025_all.parquet')
""").df())

       cnt
0  2590644


In [ ]:
# 월별 이상치 있는지 확인
print(con.execute("""
SELECT MIN(ETL_YMD) AS min_ymd, MAX(ETL_YMD) AS max_ymd
FROM read_parquet('../data/t22_2023_2025_all.parquet')
""").df())

    min_ymd   max_ymd
0  20230101  20251231


In [ ]:
# 년도별로 데이터가 얼마나 있는지 확인
print(con.execute("""
SELECT SUBSTR(CAST(ETL_YMD AS VARCHAR), 1, 6) AS ym, COUNT(*) AS cnt
FROM read_parquet('../data/t23_2023_2025_all.parquet')
GROUP BY 1
ORDER BY 1
""").df())

        ym     cnt
0   202301  130914
1   202302  118828
2   202303  131081
3   202304  126686
4   202305  131481
5   202306  127798
6   202307  132517
7   202308  133083
8   202309  127842
9   202310  132026
10  202311  127607
11  202312  131894
12  202401  132620
13  202402  124080
14  202403  131215
15  202404  127752
16  202405  129653
17  202406  128166
18  202407  133602
19  202408  133764
20  202409  128293
21  202410  134733
22  202411  128288
23  202412  132772
24  202501  133052
25  202502  120847
26  202503  132592
27  202504  129125
28  202505  133458
29  202506  129655
30  202507  135172
31  202508  134853
32  202509  129845
33  202510  133943
34  202511  129233
35  202512  134570


최종 날짜 변환 완료

In [ ]:
final_files = pd.DataFrame([
    ["t20_2023_2025_all.parquet", "통합", "사용 예정", "t20 통합 데이터"],
    ["t21_2023_2025_all.parquet", "통합", "사용 예정", "t21 통합 데이터"],
    ["t22_2023_2025_all.parquet", "통합", "사용 예정", "DATE 변환 후 통합"],
    ["t23_2023_2025_all.parquet", "통합", "사용 예정", "DATE 변환 후 통합"],
    ["t24_2023_2025_all.parquet", "통합", "사용 예정", "DATE 변환 후 통합"],
    ["t25_2023_2025_clean.parquet", "통합", "사용 예정", "컬럼 공백 제거"],
    ["t26_2023_2025_all_fixed.parquet", "통합", "최종 사용", "컬럼 불일치 수정 후 최종"],
    ["t27_2023_2025_all.parquet", "통합", "검토", "t27 통합 데이터"]
], columns=["파일명", "통합 여부", "사용 여부", "설명"])

final_files

## 최종 결과
- t26 파일은 컬럼명에 공백이 포함되어 있어, 공백 제거 후 parquet 형식으로 변환하였다.
- t22 ~ t24 데이터는 연도 컬럼이 BIGINT 타입으로 저장되어 있어, DATE 형식으로 변환한 후 데이터 검증 및 통합을 수행하였다.
- t25, t27은 동일한 구조를 가지므로, 하나의 통합 코드에서 파일명만 변경하여 parquet 파일로 변환 및 저장하였다.(t20,t21은 파일이 작아 all.csv로 최종 생성)
- t20 ~ t24, t27 데이터는 각각 연도 통합을 수행하여 `tXX_2023_2025_all.parquet` 형태의 최종 파일로 생성하였다.
- t25 데이터는 컬럼 공백 문제로 인해 공백 제거 후 `clean.parquet` 형태로 최종 파일을 생성하였다.
- t26 데이터는 2023년 1월부터 2025년 12월까지의 원본 파일 중 일부에서 컬럼 불일치가 발생하여, 공통 컬럼 기준으로 재정의한 후 `fixed.parquet` 형태의 최종 통합 파일을 생성하였다.